# Optuna with Scikit‑Learn Pipelines

This notebook demonstrates end‑to‑end hyperparameter optimization using **Optuna** with a reusable scikit‑learn pipeline.  
[Optuna on PyPi] https://pypi.org/project/optuna/

### Where Optuna Fits 

- Dataset arrives, customer wants a model  
  - You perform best model and scaler workbook
  - Once model and scaler identified:
    - Run Optuna to identify best hyperparameters for your model  

In [1]:
#!pip install optuna

## 1. Imports

In [2]:
import pandas as pd

from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector as selector
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.base import clone

import optuna


## 2. Load Sample Data

(Replace with your own dataset if needed.)

In [3]:

from sklearn.datasets import fetch_openml

data = fetch_openml("adult", version=2, as_frame=True)
df = data.frame

X = df.drop(columns="class")
y = df["class"]


## 3. Preprocessing

In [4]:
import numpy as np
from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector as selector
from sklearn.preprocessing import OrdinalEncoder

categorical_preprocessor = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

preprocessor = make_column_transformer(
    (
        categorical_preprocessor,
        selector(dtype_exclude=np.number),
    ),
    remainder="passthrough",
)

## 4. Baseline Pipeline

In [5]:

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", HistGradientBoostingClassifier(random_state=42)),
])


## 5. Optuna Objective Function

In [6]:

def objective(trial):
    params = {
        "classifier__max_depth": trial.suggest_int("max_depth", 3, 10),
        "classifier__learning_rate": trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True),
        "classifier__max_iter": trial.suggest_int("max_iter", 100, 400),
        "classifier__min_samples_leaf": trial.suggest_int("min_samples_leaf", 20, 100),
    }

    model_trial = clone(model)
    model_trial.set_params(**params)

    scores = cross_val_score(
        model_trial,
        X,
        y,
        cv=5,
        scoring="accuracy",
    )

    return scores.mean()


## 6. Run Optuna Study

In [7]:
%%time
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=25)

study.best_params, study.best_value


[I 2026-04-01 14:58:09,645] A new study created in memory with name: no-name-2428a923-6fd4-4830-9637-54015aecd52c
[I 2026-04-01 14:58:16,455] Trial 0 finished with value: 0.871667872238555 and parameters: {'max_depth': 7, 'learning_rate': 0.0668291062624505, 'max_iter': 101, 'min_samples_leaf': 93}. Best is trial 0 with value: 0.871667872238555.
[I 2026-04-01 14:58:20,778] Trial 1 finished with value: 0.873633375696023 and parameters: {'max_depth': 6, 'learning_rate': 0.09676671923198432, 'max_iter': 137, 'min_samples_leaf': 54}. Best is trial 1 with value: 0.873633375696023.
[I 2026-04-01 14:58:28,036] Trial 2 finished with value: 0.8154662423523105 and parameters: {'max_depth': 4, 'learning_rate': 0.0026009200697298175, 'max_iter': 300, 'min_samples_leaf': 46}. Best is trial 1 with value: 0.873633375696023.
[I 2026-04-01 14:58:40,143] Trial 3 finished with value: 0.8700094354025156 and parameters: {'max_depth': 10, 'learning_rate': 0.010045667315501819, 'max_iter': 379, 'min_samples_

CPU times: total: 11min 18s
Wall time: 3min 3s


({'max_depth': 8,
  'learning_rate': 0.038199936127027725,
  'max_iter': 268,
  'min_samples_leaf': 52},
 0.8740428677218522)

## 7. Train Final Model

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

best_model = clone(model)
best_model.set_params(**{
    f"classifier__{k}": v for k, v in study.best_params.items()
})

best_model.fit(X_train, y_train)
print(f'Best Model Score: {best_model.score(X_test, y_test):.3f}')
study.best_params


Best Model Score: 0.875


{'max_depth': 8,
 'learning_rate': 0.038199936127027725,
 'max_iter': 268,
 'min_samples_leaf': 52}

In [9]:
best_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ordinalencoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf